# 🤖 Model Comparison & Hyperparameter Tuning

## Objectives
1. Load `data/cleaned_salary_data.csv` using the identical 80/20 train-test split (`random_state=42`) and preprocessing pipeline (`ColumnTransformer`) as `03_baseline.ipynb`.
2. Conduct **5-fold cross-validation on the TRAINING set only** for three model architectures trained on `np.log(Salary)`:
   - `Ridge`
   - `RandomForestRegressor`
   - `HistGradientBoostingRegressor`
   Report the mean and standard deviation of CV MAE in dollars (converted via `np.exp`). The test set is strictly untouched during CV.
3. Tune the best-performing architecture using `RandomizedSearchCV` ($n\_iter = 20$, 5-fold CV on training data only) to minimize dollar MAE.
4. Evaluate the tuned model once on the unseen test set and compile a comparison table (`Median Baseline`, `Ridge`, `Tuned Model`).
5. Evaluate per-country test MAE across key focus countries (`India`, `Brazil`, `Poland`, `Germany`, `United Kingdom of Great Britain and Northern Ireland`, `United States of America`).


In [1]:
# 1. Imports and Environment Setup
import os
import sys
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, KFold, RandomizedSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score, make_scorer

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

print("Environment setup complete.")


Environment setup complete.


### 🔍 Setup Note
Libraries imported and display settings configured for clear numerical reporting.


In [2]:
# 2. Data Loading & Exact Train/Test Split (Matching 03_baseline)
data_path = os.path.join('..', 'data', 'cleaned_salary_data.csv')
df = pd.read_csv(data_path)

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")

X = df.drop(columns=['Salary'])
y = df['Salary']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")

# Define identical ColumnTransformer
cat_cols = ['Country', 'Education', 'Role', 'RemoteWork', 'OrgSize']
num_cols = ['Experience_Years', 'Work_Experience_Years']

preprocessor = ColumnTransformer([
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols),
    ('num', StandardScaler(), num_cols)
])

print(f"Categorical features: {cat_cols}")
print(f"Numeric features:     {num_cols}")


Loaded dataset: 16954 rows, 8 columns
X_train shape: (13563, 7)
X_test shape:  (3391, 7)
y_train shape: (13563,)
y_test shape:  (3391,)
Categorical features: ['Country', 'Education', 'Role', 'RemoteWork', 'OrgSize']
Numeric features:     ['Experience_Years', 'Work_Experience_Years']


### 🔍 Train/Test Split Findings
- Dataset loaded: 16,954 rows and 8 columns.
- With `test_size=0.2` and `random_state=42`:
  - `X_train shape`: (13563, 7)
  - `X_test shape`: (3391, 7)
  - `y_train shape`: (13563,)
  - `y_test shape`: (3391,)
- Feature sets: 5 categorical columns (`Country`, `Education`, `Role`, `RemoteWork`, `OrgSize`) and 2 numeric columns (`Experience_Years`, `Work_Experience_Years`).


In [3]:
# 3. 5-Fold Cross-Validation on TRAINING Data Only
# All models trained on log(Salary), evaluated on dollar MAE via exp()
models = {
    'Ridge': Ridge(alpha=1.0),
    'RandomForestRegressor': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    'HistGradientBoostingRegressor': HistGradientBoostingRegressor(random_state=42)
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_summary_rows = []

for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('reg', model)])
    fold_maes = []
    
    for tr_idx, val_idx in kf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        
        # Fit on log-transformed salary
        pipe.fit(X_tr, np.log(y_tr))
        
        # Predict and convert back with exp()
        preds = np.exp(pipe.predict(X_val))
        fold_mae = mean_absolute_error(y_val, preds)
        fold_maes.append(fold_mae)
    
    mean_mae = np.mean(fold_maes)
    std_mae = np.std(fold_maes)
    cv_summary_rows.append({
        'Model': name,
        'CV_MAE_Mean ($)': mean_mae,
        'CV_MAE_Std ($)': std_mae
    })
    print(f"{name:30s} -> CV MAE Mean: ${mean_mae:,.2f} | Std: ${std_mae:,.2f}")

df_cv = pd.DataFrame(cv_summary_rows)
print("\n5-Fold Cross-Validation Results Table (Training Set Only):")
display(df_cv)

best_cv_model = df_cv.sort_values('CV_MAE_Mean ($)').iloc[0]['Model']
print(f"\nBest architecture selected by lowest CV MAE: {best_cv_model}")


Ridge                          -> CV MAE Mean: $29,089.67 | Std: $750.48


RandomForestRegressor          -> CV MAE Mean: $29,193.06 | Std: $608.63


HistGradientBoostingRegressor  -> CV MAE Mean: $27,243.17 | Std: $543.78

5-Fold Cross-Validation Results Table (Training Set Only):


,Model,CV_MAE_Mean ($),CV_MAE_Std ($)
0,Ridge,"29,089.6739",750.4810
1,RandomForestRegressor,"29,193.0625",608.6316
2,HistGradientBoostingRegressor,"27,243.1701",543.7846



Best architecture selected by lowest CV MAE: HistGradientBoostingRegressor


### 🔍 5-Fold Cross-Validation Findings
Results on the training set (from the table directly above):
- `Ridge`: CV_MAE_Mean ($) = 29,089.6739 | CV_MAE_Std ($) = 750.4810
- `RandomForestRegressor`: CV_MAE_Mean ($) = 29,193.0625 | CV_MAE_Std ($) = 608.6316
- `HistGradientBoostingRegressor`: CV_MAE_Mean ($) = 27,243.1701 | CV_MAE_Std ($) = 543.7846

`HistGradientBoostingRegressor` achieves the lowest mean CV MAE ($27,243.1701) and lowest standard deviation ($543.7846) across the 5 validation folds, outperforming both Ridge ($29,089.6739) and Random Forest ($29,193.0625). It is selected for hyperparameter tuning.


In [4]:
# 4. Hyperparameter Tuning with RandomizedSearchCV (n_iter=20, cv=5, Training Data Only)
best_pipe = Pipeline([
    ('prep', preprocessor),
    ('reg', HistGradientBoostingRegressor(random_state=42))
])

param_dist = {
    'reg__learning_rate': [0.01, 0.03, 0.05, 0.1, 0.15],
    'reg__max_iter': [100, 150, 200, 300],
    'reg__max_leaf_nodes': [15, 31, 63, 127],
    'reg__min_samples_leaf': [10, 20, 30, 50],
    'reg__l2_regularization': [0.0, 0.1, 1.0, 10.0]
}

# Custom dollar MAE scorer on log target
def dollar_mae_scorer(y_true_log, y_pred_log):
    return -mean_absolute_error(np.exp(y_true_log), np.exp(y_pred_log))

dollar_scorer = make_scorer(dollar_mae_scorer, greater_is_better=True)

search = RandomizedSearchCV(
    best_pipe,
    param_distributions=param_dist,
    n_iter=20,
    cv=5,
    scoring=dollar_scorer,
    random_state=42,
    n_jobs=-1
)

search.fit(X_train, np.log(y_train))

best_cv_dollar_mae = -search.best_score_
print("RandomizedSearchCV Tuning Results:")
print("Best Hyperparameters:")
for param, val in search.best_params_.items():
    print(f"  {param}: {val}")

print(f"\nBest 5-Fold CV Dollar MAE: ${best_cv_dollar_mae:,.2f}")


RandomizedSearchCV Tuning Results:
Best Hyperparameters:
  reg__min_samples_leaf: 50
  reg__max_leaf_nodes: 15
  reg__max_iter: 100
  reg__learning_rate: 0.15
  reg__l2_regularization: 1.0

Best 5-Fold CV Dollar MAE: $27,286.60


### 🔍 Hyperparameter Tuning Findings
From `RandomizedSearchCV` (20 iterations, 5-fold CV on training data only):
- **Best Hyperparameters**:
  - `reg__min_samples_leaf`: 50
  - `reg__max_leaf_nodes`: 15
  - `reg__max_iter`: 100
  - `reg__learning_rate`: 0.15
  - `reg__l2_regularization`: 1.0
- **Best 5-Fold CV Dollar MAE**: $27,286.60


In [5]:
# 5. Final Evaluation on Unseen Test Set
# Evaluate tuned model
tuned_model = search.best_estimator_
y_pred_tuned_log = tuned_model.predict(X_test)
y_pred_tuned = np.exp(y_pred_tuned_log)

mae_tuned = mean_absolute_error(y_test, y_pred_tuned)
rmse_tuned = root_mean_squared_error(y_test, y_pred_tuned)
r2_tuned = r2_score(y_test, y_pred_tuned)

# Baseline model (median of y_train)
train_median = y_train.median()
y_pred_baseline = np.full_like(y_test, fill_value=train_median)
mae_base = mean_absolute_error(y_test, y_pred_baseline)
rmse_base = root_mean_squared_error(y_test, y_pred_baseline)
r2_base = r2_score(y_test, y_pred_baseline)

# Ridge model
ridge_pipe = Pipeline([('prep', preprocessor), ('reg', Ridge(alpha=1.0))])
ridge_pipe.fit(X_train, np.log(y_train))
y_pred_ridge = np.exp(ridge_pipe.predict(X_test))
mae_ridge = mean_absolute_error(y_test, y_pred_ridge)
rmse_ridge = root_mean_squared_error(y_test, y_pred_ridge)
r2_ridge = r2_score(y_test, y_pred_ridge)

final_comparison = pd.DataFrame([
    {
        'Model': 'Median Baseline',
        'MAE ($)': mae_base,
        'RMSE ($)': rmse_base,
        'R-squared': r2_base
    },
    {
        'Model': 'Ridge Regression (log-target)',
        'MAE ($)': mae_ridge,
        'RMSE ($)': rmse_ridge,
        'R-squared': r2_ridge
    },
    {
        'Model': 'Tuned HistGradientBoostingRegressor (log-target)',
        'MAE ($)': mae_tuned,
        'RMSE ($)': rmse_tuned,
        'R-squared': r2_tuned
    }
])

print("Final Model Comparison Results Table (Unseen Test Set, N = 3,391):")
display(final_comparison)


Final Model Comparison Results Table (Unseen Test Set, N = 3,391):


,Model,MAE ($),RMSE ($),R-squared
0,Median Baseline,"45,846.6697","62,496.3831",-0.0419
1,Ridge Regression (log-target),"28,513.7469","42,873.5647",0.5097
2,Tuned HistGradientBoostingRegressor (log-target),"26,329.8383","39,958.0357",0.5741


### 🔍 Final Model Evaluation Findings
Test set performance ($N = 3,391$, from table directly above):
- `Median Baseline`: MAE ($) = 45,846.6697 | RMSE ($) = 62,496.3831 | R-squared = -0.0419
- `Ridge Regression (log-target)`: MAE ($) = 28,513.7469 | RMSE ($) = 42,873.5647 | R-squared = 0.5097
- `Tuned HistGradientBoostingRegressor (log-target)`: MAE ($) = 26,329.8383 | RMSE ($) = 39,958.0357 | R-squared = 0.5741

The tuned gradient boosting model achieves the lowest test MAE ($26,329.8383) and RMSE ($39,958.0357), improving $R^2$ to 0.5741 compared to 0.5097 for Ridge regression and -0.0419 for the median baseline.


In [6]:
# 6. Test MAE Per Country for Focus Countries
focus_countries = [
    'India', 'Brazil', 'Poland', 'Germany',
    'United Kingdom of Great Britain and Northern Ireland',
    'United States of America'
]

test_analysis = X_test.copy()
test_analysis['Actual_Salary'] = y_test
test_analysis['Predicted_Salary'] = y_pred_tuned

country_eval_rows = []
for c in focus_countries:
    c_df = test_analysis[test_analysis['Country'] == c]
    c_n = len(c_df)
    c_mae = mean_absolute_error(c_df['Actual_Salary'], c_df['Predicted_Salary']) if c_n > 0 else np.nan
    country_eval_rows.append({
        'Country': c,
        'Test_Row_Count': c_n,
        'Test_MAE ($)': c_mae
    })

df_country_test_mae = pd.DataFrame(country_eval_rows)
print("Test MAE Per Country for Selected Nations:")
display(df_country_test_mae)


Test MAE Per Country for Selected Nations:


,Country,Test_Row_Count,Test_MAE ($)
0,India,102,"18,537.8936"
1,Brazil,85,"20,669.8527"
2,Poland,73,"16,916.9687"
3,Germany,345,"21,524.6080"
4,United Kingdom of Great Britain and Northern I...,235,"29,728.0604"
5,United States of America,826,"39,350.3840"


### 🔍 Per-Country Test Error Findings
Test set MAE broken down across focus countries (from table directly above):
- `India`: Test_Row_Count = 102, Test_MAE ($) = 18,537.8936
- `Brazil`: Test_Row_Count = 85, Test_MAE ($) = 20,669.8527
- `Poland`: Test_Row_Count = 73, Test_MAE ($) = 16,916.9687
- `Germany`: Test_Row_Count = 345, Test_MAE ($) = 21,524.6080
- `United Kingdom of Great Britain and Northern Ireland`: Test_Row_Count = 235, Test_MAE ($) = 29,728.0604
- `United States of America`: Test_Row_Count = 826, Test_MAE ($) = 39,350.3840

#### Observations:
The model demonstrates substantially lower dollar errors in Poland ($16,916.9687), India ($18,537.8936), Brazil ($20,669.8527), and Germany ($21,524.6080) compared to the United States ($39,350.3840). This aligns with the larger nominal scale and broader compensation spread present in the US market.
